<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/02_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Merge datasets

Combines every dataset in `MyDrive/clear_vision/datasets/raw/` into one YOLO dataset at
`MyDrive/clear_vision/datasets/merged/`. It runs `scripts/merge_datasets.py` from the repo.

- The class list is `configs/classes.yaml`, and each source has a mapping in
  `configs/mappings/<name>.yaml`. To change a mapping, edit it in the repo and push,
  then run this notebook with `FORCE = True`.
- The merge **fails** if a source has a class its mapping file doesn't list, so no boxes
  are dropped silently.
- The dataset is built on the Colab disk (`/content/merged_tmp`) and then copied to Drive
  in one go. A `.done` marker means it's finished, and re-running skips it.
- **Append mode** (`APPEND = True`) adds `NEW_SOURCES` to the existing merged dataset
  without rebuilding it:
  - Section 2 first downloads `DOWNLOAD_NEW` to `/content/raw`, in this notebook's own
    runtime. Each Colab notebook has its own runtime, so files that notebook 01 puts
    in `/content/raw` are not visible here.
  - New images are de-duplicated against the merged ones, split on their own, resized
    to `MAX_SIDE` px (JPEG q90), and copied only if Drive has room.
  - The same run re-maps the labels of any merged source whose mapping file changed.
    For example, change `pedestrian-crossing: drop` to `crosswalk` in
    `configs/mappings/rf_crosswalk.yaml`, push, then run with `APPEND = True`.
- No GPU is needed. Run all cells.


In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
WORK = "/content/merged_tmp"  # local build folder, copied to Drive at the end
SEED = 42
FORCE = False  # True rebuilds datasets/merged/ even if it is already done

# Append mode: add new sources to the existing datasets/merged/ instead of rebuilding.
APPEND = False
NEW_SOURCES = ["rf_sidewalk", "rf_barricade"]
DOWNLOAD_NEW = ["rf_sidewalk", "rf_barricade"]  # downloaded to LOCAL_RAW first; [] = skip
LOCAL_RAW = "/content/raw"  # where notebook 01 downloads with RAW_TARGET = "local"
APPEND_WORK = "/content/append_tmp"
MAX_SIDE = 1280  # new images are resized so the longer side is at most this (JPEG q90)


## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import logging
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics roboflow

sys.path.insert(0, str(REPO_DIR / "scripts"))
import download_datasets  # noqa: E402
import merge_datasets  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

RAW = Path(DRIVE_ROOT) / "datasets" / "raw"
MERGED = Path(DRIVE_ROOT) / "datasets" / "merged"
print(f"raw:    {RAW}")
print(f"merged: {MERGED}")


## 2. Download new sources

Downloads `DOWNLOAD_NEW` (Roboflow, listed in `configs/datasets.yaml`) to `LOCAL_RAW` with the same code as notebook 01. Finished downloads (with `.done`) are skipped. Then it prints classes missing from their mapping files.

In [ ]:
# 2. Download new sources to this runtime's disk (LOCAL_RAW), before the append
if DOWNLOAD_NEW:
    new_datasets = download_datasets.select_datasets(
        download_datasets.load_roboflow_datasets(REPO_DIR / "configs" / "datasets.yaml"), DOWNLOAD_NEW
    )
    failures = download_datasets.download_roboflow_datasets(
        new_datasets, Path(LOCAL_RAW), download_datasets.colab_secret("ROBOFLOW_API_KEY")
    )
    download_datasets.check_mappings(
        Path(LOCAL_RAW), DOWNLOAD_NEW, download_datasets.mappings_from_dir(REPO_DIR / "configs" / "mappings")
    )
    if failures:
        raise RuntimeError(f"Downloads failed, fix them before appending: {failures}")
else:
    print("DOWNLOAD_NEW is empty: nothing to download")


## 3. Merge or append

In [ ]:
# 3. Merge or append (prints the mappings first, then progress, then the class table)
import json

CLASSES_PATH = REPO_DIR / "configs" / "classes.yaml"
MAPPINGS_DIR = REPO_DIR / "configs" / "mappings"
raw_dirs = [Path(LOCAL_RAW), RAW] if Path(LOCAL_RAW).is_dir() else [RAW]  # local copy wins

if APPEND:
    report = merge_datasets.append(
        raw=raw_dirs,
        out=MERGED,
        new_sources=NEW_SOURCES,
        work=APPEND_WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        max_side=MAX_SIDE,
    )
else:
    report = merge_datasets.merge(
        raw=raw_dirs,
        out=MERGED,
        work=WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        force=FORCE,
    )
if report is None:  # nothing to do: show the saved report
    report = json.loads((MERGED / "merge_report.json").read_text())
    merge_datasets.log_summary(report)


## 4. Check with Ultralytics

In [ ]:
# 4. Check the dataset the way Ultralytics will load it for training
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset(str(MERGED / "data.yaml"))
print(f"OK: {data['nc']} classes")
print(f"train: {data['train']}")
print(f"val:   {data['val']}")
print(f"test:  {data.get('test')}")
print(f"images: {report['images']}")


## 5. Review

Decide whether `pedestrian-crossing` should map to `crosswalk` or stay dropped. To switch it, edit `configs/mappings/rf_crosswalk.yaml`, push, and run this notebook with `APPEND = True`. Only the rf_crosswalk labels are re-mapped.

In [ ]:
# 5. Review samples: pedestrian-crossing boxes (red) that are dropped for now
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for folder in sorted((MERGED / "review").glob("*")):
    files = sorted(folder.glob("*"))[:16]
    print(f"{folder.name}: {len(files)} images (red = review class, green = kept boxes)")
    if not files:
        continue
    cols = 4
    rows = (len(files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, path in zip(axes.flat, files):
        ax.imshow(mpimg.imread(path))
        ax.set_title(path.name[:40], fontsize=8)
    plt.tight_layout()
    plt.show()
